In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [ ]:
def load_profile(path):
    """Load a trtexec --exportProfile JSON into a DataFrame.
    Element [0] is a header {"count": N} (iteration count) — skip it.
    Every real entry has: name, timeMs, averageMs, medianMs, percentage."""
    with open(path) as f:
        raw = json.load(f)

    # header is the element WITHOUT a 'name' key ({"count": N})
    count = next((e["count"] for e in raw if "count" in e), None)
    entries = [e for e in raw if "name" in e]        # drops the {"count":…} header

    df = pd.DataFrame(entries)
    # enforce column order / dtypes
    df = df[["name", "timeMs", "averageMs", "medianMs", "percentage"]]
    for col in ["timeMs", "averageMs", "medianMs", "percentage"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"{Path(path).name}: {len(df)} kernels  (profiled iterations = {count})")
    return df

In [ ]:
import os
for root, dirs, files in os.walk("/workspace"):
    for f in files:
        if f.endswith(".json") and "per_kernel" in f.lower():
            print(os.path.join(root, f))

In [ ]:
QAT_PATH  = "/workspace/Per_kernel_json_file/qat_batch32_per_kernel_profile.json"
PTQ_PATH  = "/workspace/Per_kernel_json_file/ptq_int8_per_kernel_profile.json"
FP16_PATH = "/workspace/Per_kernel_json_file/fp16_per_kernel_profile.json"

df_qat  = load_profile(QAT_PATH)    # expect 245 kernels
df_ptq  = load_profile(PTQ_PATH)    # expect 189 kernels
df_fp16 = load_profile(FP16_PATH)   # expect 231 kernels  (same-arch, NO Q/DQ baseline)
df_qat.head(10)

In [ ]:
def common_columns(*dfs):
    cols = [set(d.columns) for d in dfs]
    common = sorted(set.intersection(*cols))
    print(f"Common columns ({len(common)}): {common}")
    for name, c in zip(("QAT", "PTQ", "FP16"), cols):
        extra = sorted(c - set(common))
        if extra: print(f"Only in {name}: {extra}")
    return common

common_cols = common_columns(df_qat, df_ptq, df_fp16)   # should be all 5

In [ ]:
df_qat_c  = df_qat[common_cols].copy();  df_qat_c["engine"]  = "QAT"
df_ptq_c  = df_ptq[common_cols].copy();  df_ptq_c["engine"]  = "PTQ"
df_fp16_c = df_fp16[common_cols].copy(); df_fp16_c["engine"] = "FP16"

print("QAT:", df_qat_c.shape, "| PTQ:", df_ptq_c.shape, "| FP16:", df_fp16_c.shape)

In [ ]:
def summarize(df, label):
    print(f"\n=== {label}  ({len(df)} kernels) ===")
    print(f"  sum(averageMs)   = {df['averageMs'].sum():.4f} ms   <- per-inference total")
    print(f"  sum(medianMs)    = {df['medianMs'].sum():.4f} ms")
    print(f"  sum(percentage)  = {df['percentage'].sum():.2f} %")
    print(f"  top-5 kernels by averageMs:")
    top = df.nlargest(5, "averageMs")[["name", "averageMs", "percentage"]]
    for _, r in top.iterrows():
        print(f"    {r['averageMs']:.4f} ms ({r['percentage']:.1f}%)  {r['name'][:70]}")

summarize(df_qat_c,  "QAT")
summarize(df_ptq_c,  "PTQ")
summarize(df_fp16_c, "FP16")

In [ ]:
def op_category(name):
    """Bucket each kernel by operation type, from its name."""
    n = name.lower()
    if "pwn(sigmoid" in n or "silu" in n:
        if "conv" in n:
            return "conv+SiLU (fused)"
        return "SiLU standalone"
    if "conv" in n:
        return "conv only"
    if "reformat" in n or "shuffle" in n:
        return "reformat"
    if "topk" in n or "nms" in n or "gather" in n:
        return "NMS/TopK"
    if "softmax" in n or "attn" in n or "matmul" in n:
        return "attention"
    return "other"

for df in (df_qat_c, df_ptq_c, df_fp16_c):
    df["op_category"] = df["name"].apply(op_category)

def op_table(df):
    return df.groupby("op_category").agg(ms=("averageMs", "sum"), n=("averageMs", "size"))

# op-category breakdown per engine (the conv+SiLU fusion story), 3-way
op_compare = (op_table(df_qat_c).add_suffix("_QAT")
              .join(op_table(df_ptq_c).add_suffix("_PTQ"), how="outer")
              .join(op_table(df_fp16_c).add_suffix("_FP16"), how="outer")
              .fillna(0))
op_compare["Δms(QAT-PTQ)"] = op_compare["ms_QAT"] - op_compare["ms_PTQ"]
op_compare.round(4)